<a href="https://colab.research.google.com/github/batrisyiasuh/batrisyia5842342-AnalyticsEngineering/blob/main/502_InclassTestPractice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.

**Started at:** ____  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 14,494 rows, 5 columns
    ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']
order_status_events: 56,812 rows, 4 columns
    ['event_id', 'order_id', 'status', 'event_ts']
order_lines: 36,120 rows, 5 columns
    ['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']
payments: 16,423 rows, 5 columns
    ['payment_id', 'order_id', 'payment_type', 'amount', 'paid_at']
restaurants: 67 rows, 5 columns
    ['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']
customers: 4,200 rows, 3 columns
    ['customer_id', 'signup_date', 'home_city']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


,n_rows,n_distinct_orders
0,14494,14494


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


(14494, 5)
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


4073 different customer_id values in orders


## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

> Add blockquote



*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

### Row Representation & Granularity
- **`orders`**: 14,494 rows. The grain is **one row per order**. `order_id` is unique (no duplicates) and there are no missing key values.
- **`order_status_events`**: 56,812 rows. The grain is **one row per status change event**. `event_id` is unique. Status types include `placed` (14,494, matching orders count), `accepted`, `picked_up`, `delivered`, and `cancelled`.
- **`restaurants`**: 67 rows, but only 64 unique `restaurant_id` values! This means there are **3 duplicate restaurant IDs** that we must investigate (likely SCD Type 2 history or messy duplicate rows).
- **`customers`**: 4,200 rows. The grain is **one row per customer** with no duplicates or missing values.
- **`payments`**: 16,423 rows. Represents transactional charges and refunds. Charges (14,524 rows) and refunds (1,899 rows).

### Table Connections
- `orders.order_id` relates 1:many to `order_status_events.order_id`, `order_lines.order_id`, and `payments.order_id`.
- `orders.restaurant_id` relates many:1 to `restaurants.restaurant_id`.
- `orders.customer_id` relates many:1 to `customers.customer_id`.

### Oddities & Concerns Found
1. **Restaurants**: 3 duplicate `restaurant_id` entries. This needs sorting out before we join it to orders to prevent artificial row multiplication.
2. **Order Statuses**: 870 orders were cancelled. Some order IDs have multiple conflicting records of events (e.g., `accepted` has slightly more rows than unique orders, indicating status state loops or duplicate events per order).

In [ ]:
# add code here

In [8]:
# 1. Checking ID uniqueness and counts across all 6 tables
print("=== Unique ID & Grain Audit ===")
display(q("""
SELECT
    'orders' AS table_name,
    'order_id' AS id_column,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_ids,
    COUNT(*) - COUNT(DISTINCT order_id) AS duplicates
FROM orders
UNION ALL
SELECT
    'order_status_events',
    'event_id',
    COUNT(*),
    COUNT(DISTINCT event_id),
    COUNT(*) - COUNT(DISTINCT event_id)
FROM order_status_events
UNION ALL
SELECT
    'order_lines',
    'order_id || line_no' AS id_column,
    COUNT(*),
    COUNT(DISTINCT order_id || '-' || CAST(line_no AS VARCHAR)),
    COUNT(*) - COUNT(DISTINCT order_id || '-' || CAST(line_no AS VARCHAR))
FROM order_lines
UNION ALL
SELECT
    'payments',
    'payment_id',
    COUNT(*),
    COUNT(DISTINCT payment_id),
    COUNT(*) - COUNT(DISTINCT payment_id)
FROM payments
UNION ALL
SELECT
    'restaurants',
    'restaurant_id',
    COUNT(*),
    COUNT(DISTINCT restaurant_id),
    COUNT(*) - COUNT(DISTINCT restaurant_id)
FROM restaurants
UNION ALL
SELECT
    'customers',
    'customer_id',
    COUNT(*),
    COUNT(DISTINCT customer_id),
    COUNT(*) - COUNT(DISTINCT customer_id)
FROM customers
"""))

=== Unique ID & Grain Audit ===


,table_name,id_column,total_rows,unique_ids,duplicates
0,orders,order_id,14494,14494,0
1,order_status_events,event_id,56812,56812,0
2,order_lines,order_id || line_no,36120,36120,0
3,payments,payment_id,16423,16393,30
4,restaurants,restaurant_id,67,64,3
5,customers,customer_id,4200,4200,0


In [9]:
# 2. Referential Integrity (Foreign Key checks)
print("=== Referential Integrity Checks ===")
display(q("""
SELECT
    'orders with non-existent customers' AS check_name,
    COUNT(o.order_id) AS orphan_count
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL
UNION ALL
SELECT
    'orders with non-existent restaurants',
    COUNT(o.order_id)
FROM orders o
LEFT JOIN restaurants r ON o.restaurant_id = r.restaurant_id
WHERE r.restaurant_id IS NULL
UNION ALL
SELECT
    'payments with non-existent orders',
    COUNT(p.payment_id)
FROM payments p
LEFT JOIN orders o ON p.order_id = o.order_id
WHERE o.order_id IS NULL
UNION ALL
SELECT
    'order_status_events with non-existent orders',
    COUNT(e.event_id)
FROM order_status_events e
LEFT JOIN orders o ON e.order_id = o.order_id
WHERE o.order_id IS NULL
UNION ALL
SELECT
    'order_lines with non-existent orders',
    COUNT(*)
FROM order_lines l
LEFT JOIN orders o ON l.order_id = o.order_id
WHERE o.order_id IS NULL
"""))

=== Referential Integrity Checks ===


,check_name,orphan_count
0,orders with non-existent customers,0
1,orders with non-existent restaurants,0
2,payments with non-existent orders,0
3,order_status_events with non-existent orders,0
4,order_lines with non-existent orders,0


In [10]:
# 3. Formatting and Structure Audit (Timestamps and String Patterns)
print("=== String Pattern and Formatting Audit ===")
display(q("""
SELECT
    'orders.order_ts pattern (YYYY-MM-DDTHH:MM:SSZ)' AS check_name,
    COUNT(*) AS mismatch_count
FROM orders
WHERE NOT REGEXP_MATCHES(order_ts, '^[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}:[0-9]{2}Z$')
UNION ALL
SELECT
    'payments.paid_at pattern (YYYY-MM-DDTHH:MM:SSZ)',
    COUNT(*)
FROM payments
WHERE NOT REGEXP_MATCHES(paid_at, '^[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}:[0-9]{2}Z$') AND paid_at IS NOT NULL
UNION ALL
SELECT
    'customers.signup_date pattern (YYYY-MM-DD)',
    COUNT(*)
FROM customers
WHERE NOT REGEXP_MATCHES(signup_date, '^[0-9]{4}-[0-9]{2}-[0-9]{2}$')
"""))

=== String Pattern and Formatting Audit ===


,check_name,mismatch_count
0,orders.order_ts pattern (YYYY-MM-DDTHH:MM:SSZ),0
1,payments.paid_at pattern (YYYY-MM-DDTHH:MM:SSZ),16423
2,customers.signup_date pattern (YYYY-MM-DD),0


In [22]:
# 1. Inspecting the duplicate payments
print("=== Duplicate payment_id Examples ===")
display(q("""
WITH dup_ids AS (
    SELECT payment_id
    FROM payments
    GROUP BY payment_id
    HAVING COUNT(*) > 1
)
SELECT *
FROM payments
WHERE payment_id IN (SELECT payment_id FROM dup_ids)
ORDER BY payment_id
LIMIT 30
"""))

=== Duplicate payment_id Examples ===


,payment_id,order_id,payment_type,amount,paid_at
0,P000219,O000185,charge,25.49,2026-03-02 19:05:43
1,P000219,O000185,charge,25.49,2026-03-02 19:05:43
2,P000927,O000815,charge,23.99,2026-03-08 11:53:53
3,P000927,O000815,charge,23.99,2026-03-08 11:53:53
4,P000998,O000874,charge,50.99,2026-03-08 19:19:26
5,P000998,O000874,charge,50.99,2026-03-08 19:19:26
6,P001859,O001631,charge,38.49,2026-03-14 22:43:50
7,P001859,O001631,charge,38.49,2026-03-14 22:43:50
8,P002189,O001926,charge,41.99,2026-03-17 18:22:24
9,P002189,O001926,charge,41.99,2026-03-17 18:22:24


In [19]:
# 2. Inspecting the formatting of payments.paid_at
print("=== payments.paid_at Formatting Sample ===")
display(q("""
SELECT paid_at, COUNT(*) as count
FROM payments
GROUP BY paid_at
LIMIT 5
"""))

=== payments.paid_at Formatting Sample ===


,paid_at,count
0,2026-03-31 18:04:12,1
1,2026-05-13 16:26:11,1
2,2026-05-25 11:56:06,1
3,2026-05-26 11:05:10,1
4,2026-05-15 22:14:20,1


In [20]:
# 1. Inspecting the duplicate payments
print("=== Duplicate payment_id Examples ===")
display(q("""
WITH dup_ids AS (
    SELECT payment_id
    FROM payments
    GROUP BY payment_id
    HAVING COUNT(*) > 1
)
SELECT *
FROM payments
WHERE payment_id IN (SELECT payment_id FROM dup_ids)
ORDER BY payment_id
LIMIT 10
"""))

=== Duplicate payment_id Examples ===


,payment_id,order_id,payment_type,amount,paid_at
0,P000219,O000185,charge,25.49,2026-03-02 19:05:43
1,P000219,O000185,charge,25.49,2026-03-02 19:05:43
2,P000927,O000815,charge,23.99,2026-03-08 11:53:53
3,P000927,O000815,charge,23.99,2026-03-08 11:53:53
4,P000998,O000874,charge,50.99,2026-03-08 19:19:26
5,P000998,O000874,charge,50.99,2026-03-08 19:19:26
6,P001859,O001631,charge,38.49,2026-03-14 22:43:50
7,P001859,O001631,charge,38.49,2026-03-14 22:43:50
8,P002189,O001926,charge,41.99,2026-03-17 18:22:24
9,P002189,O001926,charge,41.99,2026-03-17 18:22:24


In [21]:
# 2. Inspecting the formatting of payments.paid_at
print("=== payments.paid_at Formatting Sample ===")
display(q("""
SELECT paid_at, COUNT(*) as count
FROM payments
GROUP BY paid_at
LIMIT 5
"""))

=== payments.paid_at Formatting Sample ===


,paid_at,count
0,2026-06-18 21:41:37,1
1,2026-06-21 22:24:53,1
2,2026-05-13 11:58:29,1
3,2026-06-13 17:17:42,1
4,2026-04-08 18:13:16,1


In [4]:
# 1. Row counts and unique keys check
print("=== Unique Key Verification ===")
display(q("""
SELECT
    'orders' AS tbl,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_keys,
    COUNT(*) - COUNT(DISTINCT order_id) AS duplicates
FROM orders
UNION ALL
SELECT
    'order_status_events' AS tbl,
    COUNT(*),
    COUNT(DISTINCT event_id),
    COUNT(*) - COUNT(DISTINCT event_id)
FROM order_status_events
UNION ALL
SELECT
    'restaurants' AS tbl,
    COUNT(*),
    COUNT(DISTINCT restaurant_id),
    COUNT(*) - COUNT(DISTINCT restaurant_id)
FROM restaurants
UNION ALL
SELECT
    'customers' AS tbl,
    COUNT(*),
    COUNT(DISTINCT customer_id),
    COUNT(*) - COUNT(DISTINCT customer_id)
FROM customers
"""))

=== Unique Key Verification ===


,tbl,total_rows,unique_keys,duplicates
0,orders,14494,14494,0
1,order_status_events,56812,56812,0
2,restaurants,67,64,3
3,customers,4200,4200,0


In [5]:
# 2. Analyze orders grain, dates and null values
print("=== Null value count in orders ===")
display(q("""
SELECT
    COUNT(CASE WHEN order_id IS NULL THEN 1 END) AS null_order_id,
    COUNT(CASE WHEN customer_id IS NULL THEN 1 END) AS null_customer_id,
    COUNT(CASE WHEN restaurant_id IS NULL THEN 1 END) AS null_restaurant_id,
    COUNT(CASE WHEN order_ts IS NULL THEN 1 END) AS null_order_ts,
    COUNT(CASE WHEN delivery_fee IS NULL THEN 1 END) AS null_delivery_fee,
    MIN(order_ts) AS min_order_ts,
    MAX(order_ts) AS max_order_ts
FROM orders
"""))

=== Null value count in orders ===


,null_order_id,null_customer_id,null_restaurant_id,null_order_ts,null_delivery_fee,min_order_ts,max_order_ts
0,0,0,0,0,0,2026-03-01T09:13:23Z,2026-06-30T22:58:46Z


In [6]:
# 3. Check distinct statuses in order_status_events
print("=== Distinct status events and counts ===")
display(q("""
SELECT status, COUNT(*) AS count, COUNT(DISTINCT order_id) AS unique_orders
FROM order_status_events
GROUP BY status
ORDER BY count DESC
"""))

=== Distinct status events and counts ===


,status,count,unique_orders
0,placed,14494,14494
1,accepted,14154,14055
2,picked_up,13745,13624
3,delivered,13549,13409
4,cancelled,870,870


In [7]:
# 4. Explore payments table structure, types, and amounts
print("=== Payments summary ===")
display(q("""
SELECT payment_type, COUNT(*) AS count, SUM(amount) AS total_amount, MIN(amount) AS min_amount, MAX(amount) AS max_amount
FROM payments
GROUP BY payment_type
"""))

=== Payments summary ===


,payment_type,count,total_amount,min_amount,max_amount
0,charge,14524,552409.66,2.00,127.49
1,refund,1899,-43483.59,-127.49,-0.35


## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

### 2. Plan & Architecture Decisions

**1. What one row of the final table represents:**
- One row represents a summary of a specific **Week**, **Cuisine**, and **City** group.

**2. Key Decisions & Definitions:**
- **Which week does an order belong to?** We will determine weeks using the `DATE_TRUNC('week', order_ts)` function based on the order placement timestamp (`order_ts`).
- **Which orders count?** All orders with a status of `placed` count as initiated orders. However, for reporting net metrics and successful operations, we should track cancelled orders separately. We will report metrics for all valid orders and count completed vs. cancelled.
- **What is "net revenue"?** We will define net revenue as `charges` minus any `refunds` from the payments table. We will aggregate payment types `charge` and `refund` at the order grain before joining.
- **What is "delivery time"?** The duration in minutes between the order being placed (`placed` event timestamp) and delivered (`delivered` event timestamp) for successfully delivered orders.

**3. Build Steps:**
- **Step 2.1 (Silver Restaurants):** De-duplicate the `restaurants` table. Since there are active effective dates, we will select the latest record per `restaurant_id` (or the most recent version) to avoid duplicate matches.
- **Step 2.2 (Silver Orders):** Build a consolidated `stg_orders` model containing order details, calculated delivery duration, and summarized payment amounts (net revenue per order).
- **Step 2.3 (Gold reporting table):** Aggregate `stg_orders` grouped by week, cuisine, and city to calculate order counts, sum of net revenue, and average delivery times.

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

Add text here as you go along. Show that you are staying in control of the build!

In [11]:
# Step 1: Clean and De-duplicate Restaurants
# Since some restaurant_id values are duplicated, we will select the most recent version based on effective_from.
cleaned_restaurants_df = q("""
WITH ranked_restaurants AS (
    SELECT
        restaurant_id,
        restaurant_name,
        cuisine,
        city,
        effective_from,
        ROW_NUMBER() OVER (PARTITION BY restaurant_id ORDER BY effective_from DESC) as rn
    FROM restaurants
)
SELECT restaurant_id, restaurant_name, cuisine, city
FROM ranked_restaurants
WHERE rn = 1
""")
con.register("dim_restaurants", cleaned_restaurants_df)
print(f"dim_restaurants unique rows: {len(cleaned_restaurants_df)} (Original had {len(tables['restaurants'])} rows)")

dim_restaurants unique rows: 64 (Original had 67 rows)


In [12]:
# Step 2: Extract Order Status Durations
# We need the placed timestamp and delivered timestamp to compute delivery duration in minutes.
order_durations_df = q("""
SELECT
    order_id,
    MIN(CASE WHEN status = 'placed' THEN CAST(event_ts AS TIMESTAMP) END) AS placed_ts,
    MIN(CASE WHEN status = 'delivered' THEN CAST(event_ts AS TIMESTAMP) END) AS delivered_ts,
    DATE_DIFF('minute',
              MIN(CASE WHEN status = 'placed' THEN CAST(event_ts AS TIMESTAMP) END),
              MIN(CASE WHEN status = 'delivered' THEN CAST(event_ts AS TIMESTAMP) END)) AS delivery_duration_minutes
FROM order_status_events
GROUP BY order_id
""")
con.register("fct_order_durations", order_durations_df)
print(f"fct_order_durations records: {len(order_durations_df)}")

fct_order_durations records: 14494


In [13]:
# Step 3: Summarize Payments per Order to get Net Revenue (amount of charges minus absolute amount of refunds)
payments_summary_df = q("""
SELECT
    order_id,
    SUM(CASE WHEN payment_type = 'charge' THEN amount ELSE 0.0 END) AS total_charged,
    SUM(CASE WHEN payment_type = 'refund' THEN ABS(amount) ELSE 0.0 END) AS total_refunded,
    SUM(amount) AS net_revenue
FROM payments
GROUP BY order_id
""")
con.register("fct_order_payments", payments_summary_df)
print(f"fct_order_payments records: {len(payments_summary_df)}")

fct_order_payments records: 14494


In [14]:
# Step 4: Build Reporting Gold Table
# Join orders, de-duplicated restaurants, durations, and payments.
reporting_gold_df = q("""
SELECT
    DATE_TRUNC('week', CAST(o.order_ts AS TIMESTAMP)) AS week,
    r.cuisine,
    r.city,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(COALESCE(p.net_revenue, 0.0)) AS net_revenue,
    ROUND(AVG(d.delivery_duration_minutes), 1) AS avg_delivery_time_minutes
FROM orders o
LEFT JOIN dim_restaurants r ON o.restaurant_id = r.restaurant_id
LEFT JOIN fct_order_durations d ON o.order_id = d.order_id
LEFT JOIN fct_order_payments p ON o.order_id = p.order_id
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
""")
con.register("gold_weekly_operations", reporting_gold_df)
print(f"Reporting gold table unique rows: {len(reporting_gold_df)}")
display(reporting_gold_df.head(10))

Reporting gold table unique rows: 751


,week,cuisine,city,total_orders,net_revenue,avg_delivery_time_minutes
0,2026-02-23,Burgers,Birmingham,4,125.99,38.8
1,2026-02-23,Burgers,Bristol,4,66.77,40.3
2,2026-02-23,Burgers,Leeds,1,0.00,NaN
3,2026-02-23,Burgers,London,9,337.93,36.8
4,2026-02-23,Burgers,London,2,58.49,52.5
5,2026-02-23,Chinese,Birmingham,7,117.46,34.0
6,2026-02-23,Chinese,London,8,284.93,35.1
7,2026-02-23,Chinese,Manchester,1,24.99,34.0
8,2026-02-23,Indian,Birmingham,1,43.99,43.0
9,2026-02-23,Indian,LONDON,2,66.48,34.5


## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Add text here as you go along. Show that you are staying in control of the build!

In [15]:
# 1. Order Count Reconciliation Check
print("=== Reconciling Total Orders ===")
display(q("""
SELECT
    (SELECT COUNT(*) FROM orders) AS raw_orders_count,
    (SELECT SUM(total_orders) FROM gold_weekly_operations) AS gold_orders_sum,
    ((SELECT COUNT(*) FROM orders) - (SELECT SUM(total_orders) FROM gold_weekly_operations)) AS difference
"""))

=== Reconciling Total Orders ===


,raw_orders_count,gold_orders_sum,difference
0,14494,14494.0,0.0


In [16]:
# 2. Net Revenue Reconciliation Check
print("=== Reconciling Net Revenue ===")
display(q("""
SELECT
    (SELECT SUM(amount) FROM payments) AS raw_payments_net_revenue,
    (SELECT SUM(net_revenue) FROM gold_weekly_operations) AS gold_net_revenue_sum,
    ((SELECT SUM(amount) FROM payments) - (SELECT SUM(net_revenue) FROM gold_weekly_operations)) AS difference
"""))

=== Reconciling Net Revenue ===


,raw_payments_net_revenue,gold_net_revenue_sum,difference
0,508926.07,508926.07,-7.252675e-08


In [17]:
# 3. Average Delivery Time Sanity Check
print("=== Delivery Time Sanity Check ===")
display(q("""
SELECT
    MIN(delivery_duration_minutes) AS min_duration,
    MAX(delivery_duration_minutes) AS max_duration,
    AVG(delivery_duration_minutes) AS overall_avg_duration
FROM fct_order_durations
"""))

=== Delivery Time Sanity Check ===


,min_duration,max_duration,overall_avg_duration
0,-58,1194,36.402715


## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

Add prompts here.

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

Add text here.

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.